In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)



In [ ]:
import os
train_path = "train_features.csv"
test_path = "test_features.csv"
if not os.path.exists(train_path):
    if os.path.exists("../data/processed/train_features.csv"):
        train_path = "../data/processed/train_features.csv"
        test_path = "../data/processed/test_features.csv"
    elif os.path.exists("data/processed/train_features.csv"):
        train_path = "data/processed/train_features.csv"
        test_path = "data/processed/test_features.csv"
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)
train_df.head()

##Define the feature set

In [ ]:
categorical_cols = ["main_category", "currency", "country", "launch_weekday"]
numeric_cols = [
    "usd_goal_real", "goal_log", "campaign_duration",
    "launch_month", "launch_quarter",
    "category_frequency", "country_success_rate",
]

feature_cols = categorical_cols + numeric_cols

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test = test_df["target"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Train target balance:\n", y_train.value_counts(normalize=True).round(3))


##Preprocessing pipeline


In [2]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)


NameError: name 'categorical_cols' is not defined

## Train baseline models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    print(f"Trained: {name}")


##Evaluate: Accuracy, Precision, Recall, F1, ROC-AUC

In [ ]:
results = []

for name, pipe in fitted_pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(4)
results_df.to_csv("reports/baseline_model_comparison.csv", index=False)
results_df

### Classification report per model

In [ ]:
for name, pipe in fitted_pipelines.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    y_pred = pipe.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["failed", "successful"]))


### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pipe) in zip(axes, fitted_pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["failed", "successful"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.savefig("reports/figures/confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


### ROC curves

In [ ]:
plt.figure(figsize=(8, 6))

for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Baseline Models")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/roc_curves.png", dpi=300, bbox_inches="tight")
plt.show()

### Precision-Recall curves


In [ ]:
plt.figure(figsize=(8, 6))

for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(recall, precision, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves — Baseline Models")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/pr_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## Feature importance (Random Forest)


In [ ]:
rf_pipe = fitted_pipelines["Random Forest"]
feature_names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names, "importance": importances
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Top 15 Feature Importances — Random Forest")
plt.tight_layout()
plt.savefig("reports/figures/rf_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()


## Save the best model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]

print(f"Best model by ROC-AUC: {best_model_name}")

joblib.dump(best_pipeline, "models/baseline_best_model.pkl")
print("Saved to models/baseline_best_model.pkl")

In [ ]:
import shutil

shutil.make_archive("baseline_outputs", "zip", "reports")

try:
    from google.colab import files
    files.download("baseline_outputs.zip")
    files.download("models/baseline_best_model.pkl")
except (ImportError, ModuleNotFoundError):
    print("Archive created locally as 'baseline_outputs.zip'.")
    print("Files are stored locally in 'reports/' and 'models/'. Download skipped since not in Google Colab.")